## Goal

Lire les jeux de donnees disponibles dans `data/` et verifier qu'un batch peut etre cree avec le DataLoader du projet.

## Setup

In [1]:
from pathlib import Path
import sys

import pandas as pd
import torch

In [2]:
def find_project_root(start: Path = Path.cwd()) -> Path:
    for path in [start, *start.parents]:
        if (path / "data").exists() and (path / "supra_jpea.py").exists():
            return path
    raise FileNotFoundError("Impossible de trouver la racine du projet SUPRA-JEPA")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT, DATA_DIR

(PosixPath('/Users/baptistecaillerie/Documents/SUPRA-JEPA'),
 PosixPath('/Users/baptistecaillerie/Documents/SUPRA-JEPA/data'))

## Data

In [5]:
data_files = sorted(path.relative_to(PROJECT_ROOT) for path in DATA_DIR.rglob("*") if path.is_file())

In [ ]:
jepa_csv_path = DATA_DIR / "jepa" / "mp.csv.gz"
jepa_df = pd.read_csv(jepa_csv_path)

print(f"Lignes: {len(jepa_df):,}")
print(f"Colonnes: {list(jepa_df.columns)}")

Lignes: 92,762
Colonnes: ['material_id', 'cif', 'ef_per_atom']


,material_id,ef_per_atom
0,mp-23155,0.000000
1,mp-1246134,0.409911
2,mp-1182070,0.008550
3,mp-569358,0.549853
4,mp-1078637,0.051305


In [7]:
raw_3dsc_path = DATA_DIR / "raw" / "3DSC_MP.csv"
raw_3dsc_df = pd.read_csv(raw_3dsc_path, comment="#")

print(f"Lignes: {len(raw_3dsc_df):,}")
print(f"Colonnes: {len(raw_3dsc_df.columns)}")
raw_3dsc_df[["formula_sc", "tc", "sc_class", "material_id_2"]].head()

Lignes: 5,773
Colonnes: 92


,formula_sc,tc,sc_class,material_id_2
0,Ag0.02Ge2Pd1.98Sr1,2.64,Other,mp-978986
1,Ag0.15Sn0.85Te1,2.15,Other,mp-1883
2,Ag0.1Ge2Pd1.9Sr1,2.62,Other,mp-978986
3,Ag0.1In0.9Te1,1.20,Other,mp-2597
4,Ag0.2Ba1Si1.8,3.20,Other,mp-7275


## DataLoader

In [8]:
from crystal_matrices import create_dataloader

dataloader = create_dataloader(jepa_csv_path, batch_size=4, shuffle=False)
batch = next(iter(dataloader))

batch_summary = {
    "material_id": batch["material_id"],
    "atom_features_shape": tuple(batch["atom_features"].shape),
    "atom_mask_shape": tuple(batch["atom_mask"].shape),
    "ef_per_atom_shape": tuple(batch["ef_per_atom"].shape),
    "num_atoms": batch["num_atoms"].tolist(),
}
batch_summary

{'material_id': ['mp-23155', 'mp-1246134', 'mp-1182070', 'mp-569358'],
 'atom_features_shape': (4, 4, 109),
 'atom_mask_shape': (4, 4),
 'ef_per_atom_shape': (4,),
 'num_atoms': [1, 1, 4, 2]}

In [11]:
batch["material_id"]

['mp-23155', 'mp-1246134', 'mp-1182070', 'mp-569358']

In [ ]:
assert batch["atom_features"].ndim == 3
assert batch["atom_features"].shape[-1] == 109
assert batch["atom_mask"].shape == batch["atom_features"].shape[:2]
assert batch["ef_per_atom"].shape == (4,)

print("Batch DataLoader OK")